<a href="https://colab.research.google.com/github/e-eki/rsna-pneumonia-yolo-pipeline/blob/development/rsna_pneumonia_yolo_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# notebooks/colab_train.ipynb

In [2]:
# ============================================================
# Ячейка 1: Монтирование Google Drive
# ============================================================
# Drive нужен только для сохранения артефактов (модели, runs, отчёты).
# Основная работа идёт в /content — там быстрее I/O.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
# ============================================================
# Ячейка 2: Проверка GPU
# ============================================================
# Runtime → Change runtime type → T4 GPU
!nvidia-smi

Wed Sep 30 18:31:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True Tesla T4


In [18]:
# ============================================================
# Ячейка 3: Клонирование репозитория (ветка development)
# ============================================================
%cd /content

# Если папка уже существует (от прошлого запуска) — удалим, чтобы клон был чистым
!rm -rf rsna-pneumonia-yolo-pipeline

!git clone -b development https://github.com/e-eki/rsna-pneumonia-yolo-pipeline.git
%cd rsna-pneumonia-yolo-pipeline
!git branch --show-current

/content
Cloning into 'rsna-pneumonia-yolo-pipeline'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 37 (delta 6), reused 31 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 37.18 KiB | 12.39 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/rsna-pneumonia-yolo-pipeline
development


In [20]:
# ============================================================
# Ячейка 4: Установка зависимостей
# ============================================================
!pip install -q -r requirements.txt

In [24]:
# ============================================================
# Ячейка 5: Настройка Kaggle API через Colab Secrets
# ============================================================
# Токен хранится в Colab Secrets (🔑 на левой панели), НЕ в коде.
# Name секрета: KAGGLE_API_TOKEN, Value: KGAT_...
import os

try:
    from google.colab import userdata
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    print("✅ KAGGLE_API_TOKEN загружен из Colab Secrets")
except Exception as e:
    raise RuntimeError(
        "Не удалось получить KAGGLE_API_TOKEN. "
        "Добавьте секрет с этим именем в 🔑 на левой панели Colab."
    ) from e

# Проверка, что CLI видит токен
!kaggle datasets metadata poeticmage/rsna-pneumonia-detection-challenge-train-val

✅ KAGGLE_API_TOKEN загружен из Colab Secrets
Downloaded metadata to /content/rsna-pneumonia-yolo-pipeline/dataset-metadata.json


In [43]:
# ============================================================
# Ячейка 6: Скачивание датасета
# ============================================================
# !python src/download_data.py --config configs/config.yaml
!python -m src.download_data --config configs/config.yaml



2026-09-30 19:50:43 | INFO | __main__ | Скачивание датасета: poeticmage/rsna-pneumonia-detection-challenge-train-val
2026-09-30 19:50:43 | INFO | __main__ | Целевая директория: /content/rsna-pneumonia-yolo-pipeline/data/raw
2026-09-30 20:01:24 | INFO | __main__ | Датасет успешно скачан и распакован
2026-09-30 20:01:24 | INFO | __main__ | Готово. Данные в: data/raw


In [44]:
# --- Быстрая проверка структуры ---
!echo "train images: $(ls data/raw/rsna_yolo/images/train | wc -l)"
!echo "train labels: $(ls data/raw/rsna_yolo/labels/train | wc -l)"
!echo "train labels с боксами: $(find data/raw/rsna_yolo/labels/train -name '*.txt' -size +0c | wc -l)"
!echo "val images:   $(ls data/raw/rsna_yolo/images/val | wc -l)"
!echo "val labels с боксами: $(find data/raw/rsna_yolo/labels/val -name '*.txt' -size +0c | wc -l)"

train images: 18678
train labels: 18678
train labels с боксами: 4208
val images:   8006
val labels с боксами: 1804


In [46]:
!rm -rf data/processed
!rm -rf data/staged

In [47]:
!python -m src.prepare_yolo_dataset --config configs/config.yaml

2026-09-30 20:09:23 | INFO | __main__ | Источник: data/raw/rsna_yolo
2026-09-30 20:09:23 | INFO | __main__ | Назначение: data/processed
2026-09-30 20:09:23 | INFO | __main__ | [train] Изображений: 18678
2026-09-30 20:10:46 | INFO | __main__ | [train] positive=4208, negative=14470, all_rejected=0
2026-09-30 20:10:46 | INFO | __main__ | [train] boxes: total=6709, kept=0, rejected=1
2026-09-30 20:10:46 | INFO | __main__ | [val] Изображений: 8006
2026-09-30 20:11:21 | INFO | __main__ | [val] positive=1804, negative=6202, all_rejected=0
2026-09-30 20:11:21 | INFO | __main__ | [val] boxes: total=2846, kept=0, rejected=1
2026-09-30 20:11:21 | INFO | __main__ | data.yaml создан: data/processed/data.yaml
2026-09-30 20:11:22 | INFO | numexpr.utils | NumExpr defaulting to 2 threads.
2026-09-30 20:11:22 | INFO | __main__ | Отвергнутые боксы: data/staged/rejected_annotations.csv (2 строк)
2026-09-30 20:11:22 | INFO | __main__ | Отчёт фильтрации: data/staged/filtering_report.json
2026-09-30 20:11:22

In [48]:
# Сколько боксов реально осталось в processed
!echo "train processed labels с боксами: $(find data/processed/labels/train -name '*.txt' -size +0c | wc -l)"
!echo "train processed всего боксов:      $(cat data/processed/labels/train/*.txt | wc -l)"
!echo "val   processed labels с боксами: $(find data/processed/labels/val -name '*.txt' -size +0c | wc -l)"
!echo "val   processed всего боксов:      $(cat data/processed/labels/val/*.txt | wc -l)"

train processed labels с боксами: 4208
train processed всего боксов:      6708
val   processed labels с боксами: 1804
val   processed всего боксов:      2845


In [49]:
!ls -la data/processed
!ls data/processed/images/train | head -3
!ls data/processed/labels/train | head -3
!cat data/processed/data.yaml

total 20
drwxr-xr-x 4 root root 4096 Sep 30 20:11 .
drwxr-xr-x 5 root root 4096 Sep 30 20:09 ..
-rw-r--r-- 1 root root  120 Sep 30 20:11 data.yaml
drwxr-xr-x 4 root root 4096 Sep 30 20:10 images
drwxr-xr-x 4 root root 4096 Sep 30 20:10 labels
000924cf-0f8d-42bd-9158-1af53881a557.png
000fe35a-2649-43d4-b027-e67796d412e0.png
001031d9-f904-4a23-b3e5-2c088acd19c6.png
000924cf-0f8d-42bd-9158-1af53881a557.txt
000fe35a-2649-43d4-b027-e67796d412e0.txt
001031d9-f904-4a23-b3e5-2c088acd19c6.txt
names:
- pneumonia
nc: 1
path: /content/rsna-pneumonia-yolo-pipeline/data/processed
train: images/train
val: images/val


In [50]:
import pandas as pd
df = pd.read_csv("data/staged/rejected_annotations.csv")
print(df.to_string())

                              patientId  split          x           y       width  height    x_norm    y_norm    w_norm    h_norm          reject_reason
0  8da6abd4-8940-4696-bc43-6f0891cb7493  train  592.00000  560.000512  320.000000   464.0  0.734375  0.773438  0.312500  0.453125  extends_out_of_bounds
1  847f25e9-6983-4c5f-a1e6-f3629557f1a3    val   75.00032  816.000512  307.999744   208.0  0.223633  0.898438  0.300781  0.203125  extends_out_of_bounds


In [51]:
!python -m src.analyze_annotations --stage processed --split train
!python -m src.analyze_annotations --stage processed --split val
!python -m src.sample_and_draw --stage processed --split train --num 50 --grid
!python -m src.sample_and_draw --stage processed --split val --num 50 --grid

2026-09-30 20:16:16 | INFO | __main__ | Изображений: 18678, боксов: 6708
2026-09-30 20:16:16 | INFO | __main__ | Боксов: 6708
2026-09-30 20:16:16 | INFO | __main__ | ✅ Все YOLO-координаты в допустимых диапазонах
2026-09-30 20:16:16 | INFO | __main__ | Отчёт: data/reports/processed_train/report.json
2026-09-30 20:16:16 | INFO | __main__ | Графики: data/reports/processed_train
2026-09-30 20:16:22 | INFO | __main__ | Изображений: 8006, боксов: 2845
2026-09-30 20:16:22 | INFO | __main__ | Боксов: 2845
2026-09-30 20:16:23 | INFO | __main__ | ✅ Все YOLO-координаты в допустимых диапазонах
2026-09-30 20:16:23 | INFO | __main__ | Отчёт: data/reports/processed_val/report.json
2026-09-30 20:16:23 | INFO | __main__ | Графики: data/reports/processed_val
2026-09-30 20:16:24 | INFO | __main__ | Собрано 18678 изображений из data/processed/images/train
2026-09-30 20:16:24 | INFO | __main__ | Сэмплировано 50 из 18678
2026-09-30 20:16:25 | INFO | __main__ | Отрисовано 50 изображений в data/samples/proces

In [52]:
!python -m src.collect_rejected

2026-09-30 20:21:33 | INFO | __main__ | Отвергнутых аннотаций: 2
2026-09-30 20:21:33 | INFO | __main__ | Сохранено 2 отвергнутых изображений в data/samples/rejected
2026-09-30 20:21:33 | INFO | __main__ | Смотреть: python src/viewer.py --dir data/samples/rejected


In [ ]:
# ===== ПРОВЕРКА: что получилось в processed =====
# !python -m src.analyze_annotations --stage processed --split train
# !python -m src.analyze_annotations --stage processed --split val
# !python -m src.sample_and_draw --stage processed --split train --num 50 --grid
# !python -m src.sample_and_draw --stage processed --split val   --num 50 --grid

In [ ]:
# --- Смотрим, что отвергли ---
!python -m src.collect_rejected

In [ ]:
# ===== Обучение =====
!python -m src.train --config configs/config.yaml
!python -m src.evaluate --config configs/config.yaml

In [ ]:
# ============================================================
# ЭТАП 1: RAW — визуальная проверка исходников
# ============================================================
!python src/analyze_annotations.py --stage raw
!python src/sample_and_draw.py --stage raw --num 30 --grid
# Смотрим:
#   data/reports/raw/*.png              — статистика разметки
#   data/samples/raw/contact_sheet.jpg  — контактный лист

In [ ]:
# ============================================================
# Ячейка 7: Предобработка + фильтрация «плохой» разметки
# ============================================================
!python src/preprocess.py --config configs/config.yaml
# В логе — сводка: сколько боксов оставлено/отвергнуто,
# сколько изображений перемещено в staged/rejected_images/

# ============================================================
# ЭТАП 2: STAGED — проверяем, что данные стали чистыми
# ============================================================
!python src/analyze_annotations.py --stage staged
!python src/sample_and_draw.py --stage staged --num 50 --grid
# Смотрим:
#   data/reports/staged/*.png           — out_of_bounds должно быть 0
#   data/samples/staged/contact_sheet.jpg

# --- Смотрим, что именно отвергли ---
!python src/collect_rejected.py
# Создаётся data/samples/rejected/ с отвергнутыми снимками и красными боксами
# Локально смотреть: python src/viewer.py --dir data/samples/rejected

In [ ]:
# ============================================================
# Ячейка 8: Разделение на train/val и YOLO-разметка
# ============================================================
!python src/split_data.py --config configs/config.yaml

# ============================================================
# ЭТАП 3: PROCESSED — финальный датасет для модели
# ============================================================
!python src/analyze_annotations.py --stage processed --split train
!python src/analyze_annotations.py --stage processed --split val
!python src/sample_and_draw.py --stage processed --split train --num 50 --grid
!python src/sample_and_draw.py --stage processed --split val   --num 50 --grid

# ============================================================
# ГЛАВНАЯ проверка: RAW vs PROCESSED
# ============================================================
# Красный = raw (пиксельные координаты), зелёный = processed (YOLO).
# Прямоугольники должны точно совпадать.
!python src/compare_stages.py --num 40 --mode overlay --split train
!python src/compare_stages.py --num 40 --mode side    --split val
# Смотреть: data/samples/compare/

In [ ]:
# ============================================================
# Ячейка 9: Обучение YOLO
# ============================================================
!python src/train.py --config configs/config.yaml

In [ ]:
# ============================================================
# Ячейка 10: Оценка на валидации
# ============================================================
!python src/evaluate.py --config configs/config.yaml

In [ ]:
# ============================================================
# Ячейка 11: Сохранение результатов на Google Drive
# ============================================================
import shutil
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/rsna-pneumonia")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Модели и логи обучения
for src_dir, dst_name in [
    ("runs", "runs"),
    ("models", "models"),
    ("data/reports", "reports"),
    ("data/samples", "samples"),
]:
    src = Path(src_dir)
    if src.exists():
        shutil.copytree(src, DRIVE_ROOT / dst_name, dirs_exist_ok=True)
        print(f"✅ Сохранено: {src_dir} → {DRIVE_ROOT / dst_name}")
    else:
        print(f"⚠️ Пропущено (нет папки): {src_dir}")

print(f"\nВсе результаты на Google Drive: {DRIVE_ROOT}")


In [ ]:
# ============================================================
# Ячейка 12 (опционально): Итоговая сводка по пайплайну
# ============================================================
import json
from pathlib import Path
import pandas as pd

print("=" * 60)
print("СВОДКА ПО ПАЙПЛАЙНУ")
print("=" * 60)

# Отчёты по стадиям
for stage_dir in ["data/reports/raw", "data/reports/staged",
                  "data/reports/processed_train", "data/reports/processed_val"]:
    report = Path(stage_dir) / "report.json"
    if report.exists():
        data = json.load(open(report))
        print(f"\n[{stage_dir}]")
        print(f"  боксов:               {data.get('num_boxes')}")
        bpi = data.get("boxes_per_image", {})
        print(f"  изображений с боксами: {bpi.get('with_boxes')}")
        print(f"  изображений без боксов:{bpi.get('without_boxes')}")
        if "yolo_out_of_bounds" in data:
            print(f"  YOLO out-of-bounds:    {data['yolo_out_of_bounds']}")
        if "pixel_out_of_bounds_images" in data:
            print(f"  pixel out-of-bounds:   {data['pixel_out_of_bounds_images']}")

# Отчёт фильтрации
filt = Path("data/staged/filtering_report.json")
if filt.exists():
    data = json.load(open(filt))
    print(f"\n[Фильтрация]")
    print(f"  входных боксов:       {data.get('total_input_boxes')}")
    print(f"  оставлено:            {data.get('kept_boxes')}")
    print(f"  отвергнуто:           {data.get('rejected_boxes')}")
    print(f"  перемещено изображений: {data.get('moved_to_rejected')}")
    for r, c in data.get("rejected_by_reason", {}).items():
        if c:
            print(f"    {r}: {c}")

# Финальные метрики
metrics_file = Path("runs/train/results.csv")
if metrics_file.exists():
    df = pd.read_csv(metrics_file)
    last = df.iloc[-1]
    print(f"\n[Обучение — последняя эпоха]")
    print(f"  epoch:        {int(last.get('epoch', -1))}")
    print(f"  mAP@0.5:      {last.get('metrics/mAP50(B)', 'n/a')}")
    print(f"  mAP@0.5:0.95: {last.get('metrics/mAP50-95(B)', 'n/a')}")
    print(f"  precision:    {last.get('metrics/precision(B)', 'n/a')}")
    print(f"  recall:       {last.get('metrics/recall(B)', 'n/a')}")

print("\n" + "=" * 60)